In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
MNIST QPSK Neural Network with QPSK Inputs and Activations
Real-Valued Output Layer with Real Weights/Biases
Final Test Accuracy: 97%
═══════════════════════════════════════════════════════════════════════════════

This project implements a hybrid complex-valued QPSK neural network for
MNIST classification using quantization-aware training (QAT) with Straight
Through Estimator (STE). Input pixels are converted into strict QPSK symbols
and propagated through complex-valued fully connected layers with hard QPSK
activation quantization applied after each hidden layer. Unlike fully
quantized QPSK-output architectures, the final classification stage converts
the complex hidden representation into real-valued features and uses a
standard real-valued fully connected output layer for classification.
The network therefore preserves QPSK-constrained signal processing inside
the feature extractor while leveraging a higher-capacity real classifier head
for improved decision boundaries and training stability. Training is performed
using standard cross-entropy loss with Adam optimization, achieving a final
MNIST test accuracy of approximately 97% while maintaining OFDM-compatible
QPSK intermediate representations throughout the hidden layers.
"""

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import random, numpy as np, torch

# 1. QPSK Activation with STE

# Custom autograd function that hard-quantizes complex activations to the QPSK
# constellation in the forward pass: each component's sign is taken (zeros
# default to +1) and scaled by 1/√2, placing every output exactly on one of
# the four QPSK points. The backward pass uses a clipped STE, zeroing gradients
# for elements whose magnitude exceeds 1.0 so that saturated activations far
# outside the constellation range do not corrupt the weight updates.
class QPSKActivation(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input_tensor):
        ctx.save_for_backward(input_tensor)
        real_sign = torch.sign(input_tensor.real)
        real_sign[real_sign == 0] = 1.0
        imag_sign = torch.sign(input_tensor.imag)
        imag_sign[imag_sign == 0] = 1.0

        scale = 1.0 / 1.41421356
        return torch.complex(real_sign * scale, imag_sign * scale)

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, = ctx.saved_tensors
        grad_input_real = grad_output.real.clone()
        grad_input_real[torch.abs(input_tensor.real) > 1.0] = 0.0
        grad_input_imag = grad_output.imag.clone()
        grad_input_imag[torch.abs(input_tensor.imag) > 1.0] = 0.0
        return torch.complex(grad_input_real, grad_input_imag)

# 2. Complex Linear Layer

# A complex-valued fully-connected layer implemented with two standard real
# nn.Linear modules (fc_r and fc_i). The complex matrix-vector product
# (a+jb)(c+jd) is expanded into four real operations — real output = xr·wr −
# xi·wi, imaginary output = xr·wi + xi·wr — avoiding the need for native
# complex CUDA kernels while correctly capturing complex multiplication
# including the bias terms contributed by each sub-layer.
class ComplexLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(ComplexLinear, self).__init__()
        self.fc_r = nn.Linear(in_features, out_features)
        self.fc_i = nn.Linear(in_features, out_features)

    # Computes the complex linear transformation by splitting x into real and
    # imaginary parts, applying the four-real-matmul expansion, and reassembling
    # the result as a complex tensor.
    def forward(self, x):
        real_out = self.fc_r(x.real) - self.fc_i(x.imag)
        imag_out = self.fc_r(x.imag) + self.fc_i(x.real)
        return torch.complex(real_out, imag_out)

# 3. Sound QAT Network

# Three-layer MLP for MNIST with a hybrid complex/real architecture. The first
# two layers are complex (ComplexLinear + QPSK activation), enforcing that all
# intermediate representations are strict QPSK symbols. The third layer is a
# standard real nn.Linear that receives the QPSK hidden state flattened into
# real-valued features via view_as_real, producing ordinary float logits for
# cross-entropy training. This design keeps the backbone fully QPSK-constrained
# while using a simple real head for stable classification loss computation.
class SoundQPSKNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = ComplexLinear(392, 512)
        self.fc2 = ComplexLinear(512, 512)
        self.fc3 = nn.Linear(512 * 2, 10)   # REAL last layer
        self.qpsk = QPSKActivation.apply

    # Converts a float image batch into QPSK-constrained complex inputs
    # (alternating pixels → real/imag, sign-binarized and scaled by 1/√2),
    # passes them through two QPSK-quantized complex hidden layers, flattens
    # the complex output into a real feature vector via view_as_real, and
    # returns float logits from the real linear head.
    def forward(self, x):
        x = x.view(x.size(0), -1)

        # QPSK input
        x_bin = torch.sign(x)
        x_bin[x_bin == 0] = 1.0
        a = 1.0 / 1.41421356
        x_c = torch.complex(x_bin[:, 0::2] * a, x_bin[:, 1::2] * a)

        # QPSK hidden
        x_c = self.qpsk(self.fc1(x_c))
        x_c = self.qpsk(self.fc2(x_c))

        # complex -> real features
        x_real = torch.view_as_real(x_c).reshape(x_c.size(0), -1)  # (B, 1024)

        # real logits
        logits = self.fc3(x_real)  # (B, 10)
        return logits

# 4. Strict QPSK Evaluation Function

# Evaluates top-1 classification accuracy over a full DataLoader in eval mode
# with gradients disabled. Unlike the previous MSE-based variant, this network
# returns plain float logits directly, so classification is a simple argmax
# with no distance-decoding step required. Prints and returns accuracy as a percentage.
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            predicted = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Test Accuracy: {acc:.2f}%")
    return acc


# Builds a batch of complex target tensors for MSE-based training (retained
# here for compatibility). Each target is a (batch_size, num_classes) complex
# matrix initialized to Quadrant-3 (−1/√2 − j/√2) for all classes, with the
# ground-truth class position overwritten to Quadrant-1 (+1/√2 + j/√2).
# Note: this function is defined but not called in the current main(), which
# uses CrossEntropyLoss on real logits instead.
def get_qpsk_targets(labels, num_classes=10, device='cpu'):
    """Generates explicit QPSK coordinate targets for MSE Loss."""
    scale = 1.0 / 1.41421356
    # Initialize all to Quadrant 3 (-1 - j)
    targets = torch.complex(torch.full((labels.size(0), num_classes), -scale),
                            torch.full((labels.size(0), num_classes), -scale)).to(device)

    # Set correct class to Quadrant 1 (+1 + j)
    for i, label in enumerate(labels):
        targets[i, label] = torch.complex(torch.tensor(scale), torch.tensor(scale))

    return targets


# Entry point that sets up MNIST loaders, instantiates SoundQPSKNet, and trains
# for 10 epochs using standard CrossEntropyLoss on the real logits output by
# the float classifier head — a simpler training objective than the MSE-on-
# complex-pre-activations approach used in the previous variant. The QPSK
# constraint is enforced structurally in the hidden layers rather than through
# the loss function. After training, calls evaluate_model() for the final
# test accuracy.
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

    test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

    model = SoundQPSKNet().to(device)
    print("Train samples:", len(train_dataset))
    print("Test samples :", len(test_dataset))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 10

    for epoch in range(epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            predicted = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs} - Train Acc: {100*correct/total:.2f}%")

    print("\n--- Testing ---")
    evaluate_model(model, test_loader, device)

if __name__ == "__main__":
    main()